# LeetCode 2015 — Average Height of Buildings in Each Segment bằng Python

Link đề: https://leetcode.com/problems/average-height-of-buildings-in-each-segment/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Sorting, Heap (Priority Queue), Line Sweep

> **Bài Premium:** đề trong notebook được viết lại theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, nên đối chiếu lại câu chữ và ví dụ với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def averageHeightOfBuildings(self, buildings: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Làm quen kỹ thuật **quét đường thẳng** (line sweep) với **sự kiện** "tòa nhà bắt đầu / kết thúc" và **mảng hiệu** trên tọa độ thưa (dùng `dict`).
2. Duy trì đồng thời **tổng chiều cao** và **số tòa nhà** đang phủ, để tính **trung bình chia nguyên** trên mỗi đoạn cơ bản.
3. **Gộp** các đoạn liền kề có cùng trung bình để có **số đoạn ít nhất**, đồng thời **không gộp qua khoảng trống**.
4. Hiểu khoảng **nửa mở** `[start, end)` và hệ quả: tại cùng một tọa độ, sự kiện kết thúc và bắt đầu phải được **gộp trước khi** sinh đoạn.
5. Kiểm thử với brute force quét từng ô đơn vị; so sánh đáp án theo **thứ tự bất kỳ**.

## 2. Đề bài & ví dụ

Một con phố thẳng được biểu diễn bằng trục số. `buildings[i] = [start_i, end_i, height_i]` nghĩa là có một tòa nhà cao `height_i` trên đoạn **nửa mở** `[start_i, end_i)`.

Ta muốn mô tả chiều cao trên phố bằng **ít đoạn không giao nhau nhất**: `street[j] = [left_j, right_j, average_j]` nghĩa là trên `[left_j, right_j)`, **trung bình** chiều cao các tòa nhà (tổng chia **nguyên** cho số tòa) là `average_j`. Bỏ qua các vùng **không có** tòa nhà. Có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1:** `[[1,4,2],[3,9,4]]` → `[[1,3,2],[3,4,3],[4,9,4]]`.

**Ví dụ 2:** `[[1,3,2],[2,5,3],[2,8,3]]` → `[[1,3,2],[3,8,3]]`. Trên `[2,3)` có ba tòa: `(2+3+3)//3 = 2`, trùng với `[1,2)` nên gộp thành `[1,3)`. Tương tự `[3,5)` và `[5,8)` đều là 3, gộp thành `[3,8)`.

**Ví dụ 3:** `[[1,2,1],[5,6,1]]` → `[[1,2,1],[5,6,1]]` — **không** gộp vì giữa hai đoạn có khoảng trống `[2,5)`.

**Ràng buộc:** `1 ≤ len(buildings) ≤ 10⁵`, `0 ≤ start_i < end_i ≤ 10⁸`, `1 ≤ height_i ≤ 10⁵`.

In [1]:
import random
import time
import heapq
from collections import defaultdict
from itertools import groupby
from typing import List

examples = [
    ([[1, 4, 2], [3, 9, 4]], [[1, 3, 2], [3, 4, 3], [4, 9, 4]]),
    ([[1, 3, 2], [2, 5, 3], [2, 8, 3]], [[1, 3, 2], [3, 8, 3]]),
    ([[1, 2, 1], [5, 6, 1]], [[1, 2, 1], [5, 6, 1]]),
]
def same_answer(a, b):
    """Đáp án theo thứ tự bất kỳ: so sánh sau khi sắp xếp."""
    return sorted(map(list, a)) == sorted(map(list, b))

for b, exp in examples:
    print(b, "->", exp)

[[1, 4, 2], [3, 9, 4]] -> [[1, 3, 2], [3, 4, 3], [4, 9, 4]]
[[1, 3, 2], [2, 5, 3], [2, 8, 3]] -> [[1, 3, 2], [3, 8, 3]]
[[1, 2, 1], [5, 6, 1]] -> [[1, 2, 1], [5, 6, 1]]


## 3. Kiến thức nền

### 3.1 Khoảng nửa mở `[a, b)`

`[a, b)` chứa `a` nhưng **không** chứa `b`, độ dài `b − a`. Hai tòa `[1,3)` và `[3,5)` **không** chồng nhau (chạm nhau tại 3). Trên số nguyên, `[a, b)` tương ứng `range(a, b)`.

In [2]:
print(list(range(1, 3)), list(range(3, 5)), "| giao nhau:", set(range(1, 3)) & set(range(3, 5)))

[1, 2] [3, 4] | giao nhau: set()


### 3.2 Mảng hiệu (difference array) trên tọa độ thưa

Muốn "cộng `h` cho cả đoạn `[s, e)`": ghi `diff[s] += h`, `diff[e] -= h`. Đi từ trái sang phải và cộng dồn `diff` ta được giá trị tại mỗi điểm. Vì tọa độ tới `10⁸`, dùng `dict` chỉ lưu các điểm có sự kiện, rồi **sắp xếp khóa**.

In [3]:
diff = defaultdict(int)
for s, e, h in [[1, 4, 2], [3, 9, 4]]:
    diff[s] += h; diff[e] -= h
run = 0
for x in sorted(diff):
    run += diff[x]
    print(f"từ x={x}: tổng chiều cao = {run}")

từ x=1: tổng chiều cao = 2
từ x=3: tổng chiều cao = 6
từ x=4: tổng chiều cao = 4
từ x=9: tổng chiều cao = 0


### 3.3 Trung bình chia nguyên `//`

Đề dùng **chia nguyên**: `(2+3+3) // 3 = 2` (không phải `2.67`). Điều này ảnh hưởng tới việc **gộp đoạn**.

In [4]:
print((2 + 3 + 3) // 3, (2 + 3 + 3) / 3, (3 + 3) // 2)

2 2.6666666666666665 3


### 3.4 Gộp các đoạn liền kề cùng giá trị

Duyệt các đoạn theo thứ tự; nếu đoạn mới **bắt đầu đúng tại** điểm kết thúc của đoạn cuối trong kết quả **và** cùng giá trị ⇒ kéo dài đoạn cuối; ngược lại thêm đoạn mới.

In [5]:
def merge_segments(segs):
    res = []
    for l, r, v in segs:
        if res and res[-1][1] == l and res[-1][2] == v:
            res[-1][1] = r
        else:
            res.append([l, r, v])
    return res

print(merge_segments([[1, 2, 2], [2, 3, 2], [3, 5, 3], [5, 8, 3], [9, 10, 3]]))

[[1, 3, 2], [3, 8, 3], [9, 10, 3]]


## 4. Ý tưởng nền tảng / trực giác

Các **điểm sự kiện** (start hoặc end của tòa nào đó) chia trục số thành các **đoạn cơ bản**; trên mỗi đoạn cơ bản, tập tòa nhà đang phủ **không đổi** ⇒ trung bình không đổi.

Tính tay ví dụ 2 `[[1,3,2],[2,5,3],[2,8,3]]`. Điểm sự kiện: `1, 2, 3, 5, 8`.

| Đoạn cơ bản | Tòa đang phủ | Tổng | Số tòa | TB `//` |
|---|---|---|---|---|
| [1, 2) | 2 | 2 | 1 | 2 |
| [2, 3) | 2, 3, 3 | 8 | 3 | 2 |
| [3, 5) | 3, 3 | 6 | 2 | 3 |
| [5, 8) | 3 | 3 | 1 | 3 |

Gộp các đoạn liền kề cùng TB: `[1,3,2]`, `[3,8,3]` ✓.

In [6]:
b = [[1, 3, 2], [2, 5, 3], [2, 8, 3]]
pts = sorted({x for s, e, _ in b for x in (s, e)})
for l, r in zip(pts, pts[1:]):
    hs = [h for s, e, h in b if s <= l and r <= e]
    print(f"[{l},{r}): các tòa {hs} -> TB = {sum(hs) // len(hs) if hs else '-'}")

[1,2): các tòa [2] -> TB = 2
[2,3): các tòa [2, 3, 3] -> TB = 2
[3,5): các tòa [3, 3] -> TB = 3
[5,8): các tòa [3] -> TB = 3


## 5. Cách 1 — Brute Force / baseline: quét từng ô đơn vị

**Ý tưởng:** với mỗi số nguyên `x` từ `min(start)` tới `max(end) − 1`, ô `[x, x+1)` có các tòa phủ nó là các tòa có `start ≤ x < end`. Tính TB của ô (nếu có tòa), rồi gộp các ô liền kề cùng TB.

In [7]:
def avg_brute_cells(buildings):
    lo = min(s for s, _, _ in buildings); hi = max(e for _, e, _ in buildings)
    segs = []
    for x in range(lo, hi):
        hs = [h for s, e, h in buildings if s <= x < e]
        if hs:
            segs.append([x, x + 1, sum(hs) // len(hs)])
    return merge_segments(segs)

for b, exp in examples:
    got = avg_brute_cells(b); print(got, "| đúng?", same_answer(got, exp)); assert same_answer(got, exp)

[[1, 3, 2], [3, 4, 3], [4, 9, 4]] | đúng? True
[[1, 3, 2], [3, 8, 3]] | đúng? True
[[1, 2, 1], [5, 6, 1]] | đúng? True


**Độ phức tạp:** `O(L · m)` với `L = max(end) − min(start)` (có thể `10⁸`) và `m` = số tòa — **không thể** chạy với ràng buộc thật. Chỉ dùng làm chuẩn kiểm tra với tọa độ nhỏ.

**Nút thắt:** (1) quét từng ô trong khi giá trị chỉ đổi tại các điểm sự kiện; (2) với mỗi ô lại duyệt mọi tòa.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — nén tọa độ:** chỉ cần xét các điểm sự kiện; giữa hai điểm sự kiện liên tiếp, trung bình không đổi. Có `≤ 2m` điểm ⇒ `≤ 2m − 1` đoạn cơ bản.

**Insight 2 — quét với hai mảng hiệu:** tại mỗi điểm `x`: `dsum[x]` = thay đổi của tổng chiều cao, `dcnt[x]` = thay đổi của số tòa. Đi qua các điểm theo thứ tự tăng, giữ `s` (tổng) và `c` (số tòa) **của đoạn đang mở** từ điểm trước tới điểm hiện tại.

**Quy tắc "nếu… thì…":**
- Tại điểm `x` (trước khi áp dụng thay đổi ở `x`): **nếu** `c > 0` **thì** đoạn `[prev, x)` có TB `s // c` ⇒ thêm/gộp vào kết quả.
- **Nếu** `c == 0` trên `[prev, x)` **thì** đó là khoảng trống ⇒ không thêm, và đoạn sau **không** được gộp với đoạn trước (điều kiện `res[-1][1] == prev` tự lo việc này).
- Sau đó áp dụng `s += dsum[x]`, `c += dcnt[x]`, `prev = x`.

**Invariant:** khi xử lý điểm `x`, `(s, c)` là tổng và số tòa trên đoạn `[prev, x)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nén tọa độ + duyệt mọi tòa cho từng đoạn cơ bản (`O(m²)`)

Bước trung gian: đã nén tọa độ nhưng vẫn tính mỗi đoạn bằng cách duyệt mọi tòa.

In [8]:
def avg_compress_naive(buildings):
    pts = sorted({x for s, e, _ in buildings for x in (s, e)})
    segs = []
    for l, r in zip(pts, pts[1:]):
        total = cnt = 0
        for s, e, h in buildings:
            if s <= l and r <= e:
                total += h; cnt += 1
        if cnt:
            segs.append([l, r, total // cnt])
    return merge_segments(segs)

for b, exp in examples:
    assert same_answer(avg_compress_naive(b), exp)
print([avg_compress_naive(b) for b, _ in examples])

[[[1, 3, 2], [3, 4, 3], [4, 9, 4]], [[1, 3, 2], [3, 8, 3]], [[1, 2, 1], [5, 6, 1]]]


**Độ phức tạp Cách 2:** `O(m · m)` = `10¹⁰` với `m = 10⁵` — vẫn quá chậm, nhưng đã không phụ thuộc độ dài tọa độ.

### 7.2 Cách 3 — Quét đường thẳng với hai `dict` hiệu (bản nộp chính)

**Giải thích từng đoạn:**
1. `dsum[s] += h, dsum[e] -= h, dcnt[s] += 1, dcnt[e] -= 1`.
2. Duyệt `x` theo `sorted(dcnt)` (cùng tập khóa với `dsum`).
3. Nếu `c > 0`: đoạn `[prev, x)` có TB `s // c` ⇒ gộp hoặc thêm.
4. Cập nhật `s, c, prev`.

In [9]:
def avg_sweep(buildings):
    dsum = defaultdict(int); dcnt = defaultdict(int)
    for s, e, h in buildings:
        dsum[s] += h; dsum[e] -= h
        dcnt[s] += 1; dcnt[e] -= 1
    res = []
    total = cnt = 0
    prev = None
    for x in sorted(dcnt):
        if cnt > 0:
            avg = total // cnt
            if res and res[-1][1] == prev and res[-1][2] == avg:
                res[-1][1] = x                     # kéo dài đoạn cuối
            else:
                res.append([prev, x, avg])
        total += dsum[x]; cnt += dcnt[x]
        prev = x
    return res

for b, exp in examples:
    got = avg_sweep(b); print(got); assert same_answer(got, exp)

[[1, 3, 2], [3, 4, 3], [4, 9, 4]]
[[1, 3, 2], [3, 8, 3]]
[[1, 2, 1], [5, 6, 1]]


In [10]:
class SolutionSweep:
    def averageHeightOfBuildings(self, buildings: List[List[int]]) -> List[List[int]]:
        dsum = defaultdict(int); dcnt = defaultdict(int)
        for s, e, h in buildings:
            dsum[s] += h; dsum[e] -= h
            dcnt[s] += 1; dcnt[e] -= 1
        res, total, cnt, prev = [], 0, 0, None
        for x in sorted(dcnt):
            if cnt:
                avg = total // cnt
                if res and res[-1][1] == prev and res[-1][2] == avg:
                    res[-1][1] = x
                else:
                    res.append([prev, x, avg])
            total += dsum[x]; cnt += dcnt[x]; prev = x
        return res

print([SolutionSweep().averageHeightOfBuildings(b) for b, _ in examples])

[[[1, 3, 2], [3, 4, 3], [4, 9, 4]], [[1, 3, 2], [3, 8, 3]], [[1, 2, 1], [5, 6, 1]]]


**Độ phức tạp Cách 3:** tạo `dict` `O(m)`, sắp `≤ 2m` khóa `O(m log m)`, quét `O(m)` ⇒ `O(m log m)`. Space `O(m)`.

### 7.3 Cách 4 — Danh sách sự kiện đã sắp + `groupby`

Thay hai `dict` bằng danh sách sự kiện `(x, Δtổng, Δsố)`, sắp theo `x`, rồi **gộp các sự kiện cùng `x`** bằng `itertools.groupby` — quan trọng để không sinh đoạn rỗng `[x, x)`.

In [11]:
def avg_events(buildings):
    events = []
    for s, e, h in buildings:
        events.append((s, h, 1)); events.append((e, -h, -1))
    events.sort()
    res, total, cnt, prev = [], 0, 0, None
    for x, grp in groupby(events, key=lambda ev: ev[0]):
        if cnt:
            avg = total // cnt
            if res and res[-1][1] == prev and res[-1][2] == avg:
                res[-1][1] = x
            else:
                res.append([prev, x, avg])
        for _, dh, dc in grp:
            total += dh; cnt += dc
        prev = x
    return res

for b, exp in examples:
    assert same_answer(avg_events(b), exp)
print([avg_events(b) for b, _ in examples])

[[[1, 3, 2], [3, 4, 3], [4, 9, 4]], [[1, 3, 2], [3, 8, 3]], [[1, 2, 1], [5, 6, 1]]]


**Độ phức tạp Cách 4:** `O(m log m)` (sắp `2m` sự kiện), Space `O(m)`. Ưu điểm: dễ mở rộng khi mỗi sự kiện mang nhiều thông tin hơn.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 2. `dsum = {1:+2, 3:−2, 2:+6, 5:−3, 8:−3}`, `dcnt = {1:+1, 3:−1, 2:+2, 5:−1, 8:−1}`.

| x | (total, cnt) trên [prev, x) | đoạn sinh ra | hành động | res | (total, cnt) sau |
|---|---|---|---|---|---|
| 1 | (0, 0) | — | — | [] | (2, 1) |
| 2 | (2, 1) | [1,2) TB 2 | thêm | [[1,2,2]] | (8, 3) |
| 3 | (8, 3) | [2,3) TB 2 | **gộp** | [[1,3,2]] | (6, 2) |
| 5 | (6, 2) | [3,5) TB 3 | thêm | [[1,3,2],[3,5,3]] | (3, 1) |
| 8 | (3, 1) | [5,8) TB 3 | **gộp** | [[1,3,2],[3,8,3]] | (0, 0) |

In [12]:
def trace_sweep(buildings):
    dsum = defaultdict(int); dcnt = defaultdict(int)
    for s, e, h in buildings:
        dsum[s] += h; dsum[e] -= h; dcnt[s] += 1; dcnt[e] -= 1
    res, total, cnt, prev = [], 0, 0, None
    for x in sorted(dcnt):
        act = "(khoảng trống / chưa bắt đầu)"
        if cnt:
            avg = total // cnt
            if res and res[-1][1] == prev and res[-1][2] == avg:
                res[-1][1] = x; act = f"[{prev},{x}) TB {avg} -> gộp"
            else:
                res.append([prev, x, avg]); act = f"[{prev},{x}) TB {avg} -> thêm"
        print(f"x={x}: trên [prev,x) total={total}, cnt={cnt} | {act} | res={res}")
        total += dsum[x]; cnt += dcnt[x]; prev = x
    return res

print("Đáp án:", trace_sweep([[1, 3, 2], [2, 5, 3], [2, 8, 3]]))

x=1: trên [prev,x) total=0, cnt=0 | (khoảng trống / chưa bắt đầu) | res=[]
x=2: trên [prev,x) total=2, cnt=1 | [1,2) TB 2 -> thêm | res=[[1, 2, 2]]
x=3: trên [prev,x) total=8, cnt=3 | [2,3) TB 2 -> gộp | res=[[1, 3, 2]]
x=5: trên [prev,x) total=6, cnt=2 | [3,5) TB 3 -> thêm | res=[[1, 3, 2], [3, 5, 3]]
x=8: trên [prev,x) total=3, cnt=1 | [5,8) TB 3 -> gộp | res=[[1, 3, 2], [3, 8, 3]]
Đáp án: [[1, 3, 2], [3, 8, 3]]


In [13]:
print("Đáp án:", trace_sweep([[1, 2, 1], [5, 6, 1]]))

x=1: trên [prev,x) total=0, cnt=0 | (khoảng trống / chưa bắt đầu) | res=[]
x=2: trên [prev,x) total=1, cnt=1 | [1,2) TB 1 -> thêm | res=[[1, 2, 1]]
x=5: trên [prev,x) total=0, cnt=0 | (khoảng trống / chưa bắt đầu) | res=[[1, 2, 1]]
x=6: trên [prev,x) total=1, cnt=1 | [5,6) TB 1 -> thêm | res=[[1, 2, 1], [5, 6, 1]]
Đáp án: [[1, 2, 1], [5, 6, 1]]


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def averageHeightOfBuildings(self, buildings: List[List[int]]) -> List[List[int]]:
        dsum = defaultdict(int)          # thay đổi tổng chiều cao tại mỗi tọa độ
        dcnt = defaultdict(int)          # thay đổi số tòa nhà tại mỗi tọa độ
        for s, e, h in buildings:
            dsum[s] += h; dsum[e] -= h
            dcnt[s] += 1; dcnt[e] -= 1
        res = []
        total = cnt = 0
        prev = None
        for x in sorted(dcnt):
            if cnt:                                       # đoạn [prev, x) có tòa nhà
                avg = total // cnt
                if res and res[-1][1] == prev and res[-1][2] == avg:
                    res[-1][1] = x                        # liền kề và cùng TB: gộp
                else:
                    res.append([prev, x, avg])
            total += dsum[x]
            cnt += dcnt[x]
            prev = x
        return res

for b, exp in examples:
    assert same_answer(Solution().averageHeightOfBuildings(b), exp)
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Đoạn cơ bản đúng.** Gọi `x₀ < x₁ < ... < x_k` là các điểm sự kiện. Với mọi điểm `t ∈ [x_j, x_{j+1})`, tòa `i` phủ `t` ⇔ `start_i ≤ x_j` và `end_i ≥ x_{j+1}` (vì không có start/end nào nằm trong khoảng mở `(x_j, x_{j+1})`). Vậy trên mỗi đoạn cơ bản, tập tòa và do đó TB không đổi.
2. **Invariant `(total, cnt)`.** Sau khi áp dụng mọi thay đổi tại các điểm `≤ x_j`, `total` = tổng `h` của các tòa có `start ≤ x_j < end`, `cnt` = số tòa đó — đúng vì mỗi tòa đóng góp `+h, +1` tại start và `−h, −1` tại end.
3. **Tối thiểu số đoạn.** Mô tả hợp lệ phải tách tại mọi điểm mà TB thay đổi, và tại mọi biên giữa vùng có tòa / không có tòa. Thuật toán gộp **mọi** cặp đoạn cơ bản liền kề cùng TB, nên mỗi đoạn kết quả là một đoạn **cực đại** có TB không đổi; không thể dùng ít đoạn hơn vì hai đoạn kết quả liền kề nhau luôn khác TB, còn hai đoạn không liền kề bị ngăn bởi khoảng trống.
4. Điều kiện `res[-1][1] == prev` đảm bảo **không gộp qua khoảng trống**.

## 11. Phân tích Time / Space

Gọi `m` = số tòa nhà, `L` = độ dài vùng tọa độ.

| Cách | Time | Space |
|---|---|---|
| 1. Quét từng ô | `O(L · m)` | `O(L)` |
| 2. Nén tọa độ + duyệt tòa | `O(m²)` | `O(m)` |
| 3. Quét + 2 dict hiệu | `O(m log m)` | `O(m)` |
| 4. Sự kiện + groupby | `O(m log m)` | `O(m)` |

In [15]:
rng = random.Random(2015)
big = []
for _ in range(10**5):
    s = rng.randint(0, 10**8 - 1); e = rng.randint(s + 1, min(10**8, s + 10**6)); big.append([s, e, rng.randint(1, 10**5)])
for name, fn in [("sweep", avg_sweep), ("events", avg_events), ("Solution", lambda b: Solution().averageHeightOfBuildings(b))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:8s}: {len(r)} đoạn ({time.perf_counter() - t:.3f}s)")

sweep   : 198892 đoạn (0.322s)


events  : 198892 đoạn (0.288s)


Solution: 198892 đoạn (0.331s)


## 12. Bộ test

Ví dụ đề; biên (một tòa, các tòa chạm nhau, tòa lồng nhau, tòa trùng hệt, khoảng trống giữa hai đoạn cùng TB); đối chiếu brute force quét từng ô trên dữ liệu ngẫu nhiên tọa độ nhỏ (seed cố định). Vì đáp án là **tối thiểu** và mỗi đoạn là cực đại, đáp án là **duy nhất** (sau khi sắp xếp), nên có thể so sánh trực tiếp.

In [16]:
fns = {
    "brute": avg_brute_cells, "compress": avg_compress_naive, "sweep": avg_sweep, "events": avg_events,
    "Solution": lambda b: Solution().averageHeightOfBuildings(b),
    "SolutionSweep": lambda b: SolutionSweep().averageHeightOfBuildings(b),
}
edge = [
    ([[0, 5, 7]], [[0, 5, 7]]),
    ([[0, 2, 3], [2, 4, 3]], [[0, 4, 3]]),                 # chạm nhau, cùng chiều cao -> gộp
    ([[0, 2, 3], [2, 4, 5]], [[0, 2, 3], [2, 4, 5]]),
    ([[0, 10, 1], [3, 5, 1]], [[0, 10, 1]]),               # lồng nhau, TB không đổi
    ([[0, 10, 1], [3, 5, 4]], [[0, 3, 1], [3, 5, 2], [5, 10, 1]]),
    ([[1, 3, 2], [1, 3, 2]], [[1, 3, 2]]),                 # trùng hệt
    ([[0, 1, 5], [2, 3, 5]], [[0, 1, 5], [2, 3, 5]]),       # khoảng trống, không gộp
]
for b, exp in examples + edge:
    for name, fn in fns.items():
        assert same_answer(fn(b), exp), (name, b, fn(b))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [17]:
rng = random.Random(42)
for _ in range(500):
    b = []
    for _ in range(rng.randint(1, 8)):
        s = rng.randint(0, 12); e = rng.randint(s + 1, 14); b.append([s, e, rng.randint(1, 6)])
    want = avg_brute_cells(b)
    for name, fn in fns.items():
        assert same_answer(fn(b), want), (name, b)
print("500 test ngẫu nhiên: mọi cách khớp brute force")

500 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Gộp qua khoảng trống

Chỉ so sánh TB mà quên kiểm `res[-1][1] == prev` ⇒ ví dụ 3 bị gộp thành `[1,6,1]`, phủ cả vùng không có tòa.

In [18]:
def wrong_merge_gap(buildings):
    dsum = defaultdict(int); dcnt = defaultdict(int)
    for s, e, h in buildings:
        dsum[s] += h; dsum[e] -= h; dcnt[s] += 1; dcnt[e] -= 1
    res, total, cnt, prev = [], 0, 0, None
    for x in sorted(dcnt):
        if cnt:
            avg = total // cnt
            if res and res[-1][2] == avg:            # SAI: thiếu điều kiện liền kề
                res[-1][1] = x
            else:
                res.append([prev, x, avg])
        total += dsum[x]; cnt += dcnt[x]; prev = x
    return res

b = [[1, 2, 1], [5, 6, 1]]
print("sai:", wrong_merge_gap(b), "| đúng:", Solution().averageHeightOfBuildings(b))

sai: [[1, 6, 1]] | đúng: [[1, 2, 1], [5, 6, 1]]


In [19]:
print("sửa:", avg_sweep([[1, 2, 1], [5, 6, 1]]))

sửa: [[1, 2, 1], [5, 6, 1]]


### Lỗi 2 — Dùng chia số thực thay cho chia nguyên

`(2+3+3)/3 = 2.67 ≠ 2` ⇒ ví dụ 2 không gộp được `[1,2)` với `[2,3)` và còn trả về số thực.

In [20]:
def wrong_float(buildings):
    dsum = defaultdict(int); dcnt = defaultdict(int)
    for s, e, h in buildings:
        dsum[s] += h; dsum[e] -= h; dcnt[s] += 1; dcnt[e] -= 1
    res, total, cnt, prev = [], 0, 0, None
    for x in sorted(dcnt):
        if cnt:
            avg = total / cnt                          # SAI: chia số thực
            if res and res[-1][1] == prev and res[-1][2] == avg: res[-1][1] = x
            else: res.append([prev, x, avg])
        total += dsum[x]; cnt += dcnt[x]; prev = x
    return res

b = examples[1][0]
print("sai:", wrong_float(b), "| đúng:", examples[1][1])

sai: [[1, 2, 2.0], [2, 3, 2.6666666666666665], [3, 8, 3.0]] | đúng: [[1, 3, 2], [3, 8, 3]]


In [21]:
print("sửa (//):", avg_sweep(examples[1][0]))

sửa (//): [[1, 3, 2], [3, 8, 3]]


### Lỗi 3 — Không gộp các đoạn cùng TB (không tối thiểu)

Trả về mọi đoạn cơ bản là **đúng về giá trị** nhưng **không phải số đoạn ít nhất**.

In [22]:
def wrong_no_merge(buildings):
    dsum = defaultdict(int); dcnt = defaultdict(int)
    for s, e, h in buildings:
        dsum[s] += h; dsum[e] -= h; dcnt[s] += 1; dcnt[e] -= 1
    res, total, cnt, prev = [], 0, 0, None
    for x in sorted(dcnt):
        if cnt:
            res.append([prev, x, total // cnt])        # SAI: không gộp
        total += dsum[x]; cnt += dcnt[x]; prev = x
    return res

b = examples[1][0]
print("sai:", wrong_no_merge(b), "| đúng:", examples[1][1])

sai: [[1, 2, 2], [2, 3, 2], [3, 5, 3], [5, 8, 3]] | đúng: [[1, 3, 2], [3, 8, 3]]


In [23]:
print("sửa:", avg_sweep(examples[1][0]))

sửa: [[1, 3, 2], [3, 8, 3]]


### Lỗi 4 — Xử lý từng sự kiện riêng lẻ khi nhiều sự kiện trùng tọa độ

Với danh sách sự kiện, nếu sinh đoạn tại **mỗi** sự kiện (thay vì mỗi tọa độ), ta sẽ sinh đoạn rỗng `[x, x)`, hoặc với thứ tự sự kiện "xấu" sẽ chèn một đoạn rỗng phá vỡ việc gộp.

In [24]:
def wrong_per_event(buildings):
    events = sorted([(s, h, 1) for s, e, h in buildings] + [(e, -h, -1) for s, e, h in buildings])
    res, total, cnt, prev = [], 0, 0, None
    for x, dh, dc in events:                            # SAI: không gộp các sự kiện cùng x
        if cnt:
            avg = total // cnt
            if res and res[-1][1] == prev and res[-1][2] == avg: res[-1][1] = x
            else: res.append([prev, x, avg])
        total += dh; cnt += dc; prev = x
    return res

b = [[0, 2, 1], [2, 4, 5], [0, 4, 3]]
print("sai:", wrong_per_event(b), "| đúng:", Solution().averageHeightOfBuildings(b))

sai: [[0, 0, 1], [0, 2, 2], [2, 2, 3], [2, 4, 4], [4, 4, 3]] | đúng: [[0, 2, 2], [2, 4, 4]]


In [25]:
print("sửa (groupby theo tọa độ):", avg_events(b))

sửa (groupby theo tọa độ): [[0, 2, 2], [2, 4, 4]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhiều khoảng chồng nhau, cần thông tin "trên từng đoạn"** ⇒ quét đường thẳng: sự kiện `+` tại start, `−` tại end.
- Tọa độ lớn (`10⁸`, `10⁹`) ⇒ `dict` + `sorted(keys)` thay cho mảng.
- Cần trung bình ⇒ duy trì **hai** đại lượng cộng được (tổng, số lượng), không duy trì trung bình trực tiếp.
- "Số đoạn ít nhất" ⇒ gộp đoạn liền kề cùng giá trị, nhớ kiểm **liền kề**.
- Khoảng nửa mở ⇒ tại cùng tọa độ, áp dụng **tất cả** thay đổi trước khi sinh đoạn tiếp.

## 15. Biến thể của bài

1. **LeetCode 218 — The Skyline Problem (Hard):** chiều cao **lớn nhất** trên mỗi đoạn ⇒ quét + heap có xóa lười (bài tập 5).
2. **LeetCode 2381 — Shifting Letters II / 1094 — Car Pooling:** mảng hiệu trên tọa độ nhỏ.
3. **LeetCode 732 — My Calendar III:** số khoảng chồng nhau nhiều nhất ⇒ quét với `dcnt`.
4. **LeetCode 2158 — Amount of New Area Painted Each Day (Premium):** quét với cấu trúc tập hợp.
5. **Tổng chiều cao (không chia)** trên mỗi đoạn: bỏ `dcnt`, chỉ dùng `dsum` (nhưng vẫn cần biết vùng trống!).

Minh họa biến thể 3 (số tòa chồng nhau tối đa) với dữ liệu tự dựng:

In [26]:
def max_overlap(buildings):
    dcnt = defaultdict(int)
    for s, e, _ in buildings:
        dcnt[s] += 1; dcnt[e] -= 1
    cur = best = 0
    for x in sorted(dcnt):
        cur += dcnt[x]; best = max(best, cur)
    return best

rng = random.Random(732)
for _ in range(300):
    b = [[s, rng.randint(s + 1, 12), 1] for s in (rng.randint(0, 10) for _ in range(rng.randint(1, 8)))]
    brute = max(sum(1 for s, e, _ in b if s <= x < e) for x in range(0, 13))
    assert max_overlap(b) == brute
print("Số tòa chồng nhau tối đa khớp brute force; ví dụ 2 ->", max_overlap(examples[1][0]))

Số tòa chồng nhau tối đa khớp brute force; ví dụ 2 -> 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét từng ô | brute force trên trục | `O(L·m)` | `O(L)` | kiểm tra, tọa độ nhỏ |
| 2. Nén + duyệt tòa | đoạn cơ bản, tính lại từ đầu | `O(m²)` | `O(m)` | bước trung gian |
| 3. Quét + 2 dict | tổng & số tòa cộng dồn | `O(m log m)` | `O(m)` | **nộp bài** |
| 4. Sự kiện + groupby | danh sách sự kiện đã sắp | `O(m log m)` | `O(m)` | sự kiện mang nhiều thông tin |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `[[0,10,5]]`.
2. **Tính tay:** `[[1,5,2],[3,10,4]]` (ví dụ trong phần mô tả đề).
3. **Điền TODO:** `covered_length(street)` — tổng độ dài các đoạn trong kết quả (= độ dài phần phố có tòa nhà).
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `max_height_segments(buildings)` — mô tả phố bằng ít đoạn nhất, nhưng giá trị là chiều cao **lớn nhất** trên đoạn (bỏ vùng trống). Dùng quét + heap xóa lười; kiểm bằng brute force quét từng ô.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def covered_length(street):
    # TODO
    pass

In [28]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_average_height(buildings):
    # TODO
    pass

In [29]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def max_height_segments(buildings):
    # TODO: sự kiện theo tọa độ; heap max (-h, end); xóa lười các tòa đã kết thúc
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Một tòa. (2) Một đoạn. (3) `[[0,10,5]]`.

**Bài 2.** (1) Điểm sự kiện `1, 3, 5, 10`. (2) TB: `2`, `(2+4)//2 = 3`, `4`. (3) Không có hai đoạn liền kề cùng TB.

**Bài 3.** (1) Mỗi đoạn đóng góp `r − l`. (2) `sum(...)`. (3) Kiểm bằng cách đếm ô có tòa trong brute force.

**Bài 4.** (1) Hai `defaultdict(int)`. (2) Sinh đoạn **trước** khi áp dụng thay đổi tại `x`. (3) Gộp khi liền kề và cùng TB.

**Bài 5.** (1) Tại mỗi tọa độ `x`: đẩy các tòa bắt đầu tại `x` vào heap `(-h, end)`. (2) Lấy ra khỏi đỉnh heap các tòa có `end ≤ x`. (3) Chiều cao lớn nhất trên `[x, x_tiếp)` là `-heap[0][0]` nếu heap khác rỗng; gộp như cũ.

## 19. Đáp án tham khảo

In [30]:
# Bài 1, 2
assert Solution().averageHeightOfBuildings([[0, 10, 5]]) == [[0, 10, 5]]
assert same_answer(Solution().averageHeightOfBuildings([[1, 5, 2], [3, 10, 4]]), [[1, 3, 2], [3, 5, 3], [5, 10, 4]])
print("Bài 1, 2: OK")

Bài 1, 2: OK


In [31]:
# Bài 3, 4
def covered_length_ans(street):
    return sum(r - l for l, r, _ in street)

my_average_height_ans = avg_sweep
rng = random.Random(3)
for _ in range(200):
    b = [[s, rng.randint(s + 1, 15), rng.randint(1, 5)] for s in (rng.randint(0, 13) for _ in range(rng.randint(1, 7)))]
    cells = sum(1 for x in range(0, 16) if any(s <= x < e for s, e, _ in b))
    assert covered_length_ans(my_average_height_ans(b)) == cells
print("Bài 3, 4: OK; ví dụ 3 phủ", covered_length_ans(Solution().averageHeightOfBuildings(examples[2][0])), "đơn vị")

Bài 3, 4: OK; ví dụ 3 phủ 2 đơn vị


In [32]:
# Bài 5
def max_height_segments_ans(buildings):
    starts = defaultdict(list); pts = set()
    for s, e, h in buildings:
        starts[s].append((-h, e)); pts.add(s); pts.add(e)
    pts = sorted(pts); heap = []; res = []
    for i, x in enumerate(pts[:-1]):
        for item in starts[x]:
            heapq.heappush(heap, item)
        while heap and heap[0][1] <= x:            # xóa lười: tòa đã kết thúc
            heapq.heappop(heap)
        if heap:
            v, nx = -heap[0][0], pts[i + 1]
            if res and res[-1][1] == x and res[-1][2] == v: res[-1][1] = nx
            else: res.append([x, nx, v])
    return res

def max_height_brute(buildings):
    lo = min(s for s, _, _ in buildings); hi = max(e for _, e, _ in buildings)
    segs = [[x, x + 1, max(h for s, e, h in buildings if s <= x < e)]
            for x in range(lo, hi) if any(s <= x < e for s, e, _ in buildings)]
    return merge_segments(segs)

rng = random.Random(218)
for _ in range(400):
    b = [[s, rng.randint(s + 1, 14), rng.randint(1, 6)] for s in (rng.randint(0, 12) for _ in range(rng.randint(1, 8)))]
    assert max_height_segments_ans(b) == max_height_brute(b)
print("Bài 5: OK; ví dụ 1 ->", max_height_segments_ans(examples[0][0]))

Bài 5: OK; ví dụ 1 -> [[1, 3, 2], [3, 9, 4]]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template quét đường thẳng + gộp đoạn:**

```python
dsum, dcnt = defaultdict(int), defaultdict(int)
for s, e, h in items:
    dsum[s] += h; dsum[e] -= h; dcnt[s] += 1; dcnt[e] -= 1
res, total, cnt, prev = [], 0, 0, None
for x in sorted(dcnt):
    if cnt:                                   # đoạn [prev, x)
        v = total // cnt
        if res and res[-1][1] == prev and res[-1][2] == v: res[-1][1] = x
        else: res.append([prev, x, v])
    total += dsum[x]; cnt += dcnt[x]; prev = x
```

In [33]:
def template(items):
    dsum, dcnt = defaultdict(int), defaultdict(int)
    for s, e, h in items:
        dsum[s] += h; dsum[e] -= h; dcnt[s] += 1; dcnt[e] -= 1
    res, total, cnt, prev = [], 0, 0, None
    for x in sorted(dcnt):
        if cnt:
            v = total // cnt
            if res and res[-1][1] == prev and res[-1][2] == v: res[-1][1] = x
            else: res.append([prev, x, v])
        total += dsum[x]; cnt += dcnt[x]; prev = x
    return res
print([template(b) for b, _ in examples])

[[[1, 3, 2], [3, 4, 3], [4, 9, 4]], [[1, 3, 2], [3, 8, 3]], [[1, 2, 1], [5, 6, 1]]]


**Checklist:**

- [ ] Trung bình bằng **chia nguyên** `//`.
- [ ] Sinh đoạn `[prev, x)` **trước** khi áp dụng thay đổi tại `x`.
- [ ] Gộp chỉ khi **liền kề** (`res[-1][1] == prev`) **và** cùng TB.
- [ ] Bỏ đoạn có `cnt == 0` (khoảng trống).
- [ ] Gộp mọi sự kiện cùng tọa độ (dict tự làm; danh sách sự kiện cần `groupby`).
- [ ] Đã thử: các tòa chạm nhau, lồng nhau, trùng hệt, có khoảng trống.